# tcar_nuscenes Dataset Validation

Validates the NuScenes-formatted dataset produced by `raw2nuscenes.py`:
1. Loads it via `nuscenes-devkit`
2. Reports table sizes and scene structure
3. Verifies time synchronization between LIDAR and 7 cameras
4. Renders a few samples for visual sanity check

**Run with the `nuscenes_parser` conda env.**

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from nuscenes.nuscenes import NuScenes

DATAROOT = "/data/tcar_nuscenes"
VERSION = "v1.0-trainval"
INTERMEDIATE_ROOT = Path("/data/intermediate")

nusc = NuScenes(version=VERSION, dataroot=DATAROOT, verbose=False)
print(f"Loaded {len(nusc.scene)} scenes, {len(nusc.sample)} samples, "
      f"{len(nusc.sample_data)} sample_data, {len(nusc.ego_pose)} ego_pose")

## 1. Dataset overview

In [ ]:
for tbl in ['scene', 'sample', 'sample_data', 'ego_pose', 'sensor',
            'calibrated_sensor', 'log', 'sample_annotation', 'instance',
            'category', 'attribute', 'visibility', 'map']:
    print(f"{tbl:25}  {len(getattr(nusc, tbl)):>8}")

In [ ]:
# Per-log breakdown
for log in nusc.log:
    n_scenes = sum(1 for s in nusc.scene if s['log_token'] == log['token'])
    print(f"  {log['logfile']:30}  date={log['date_captured']}  scenes={n_scenes}")

In [ ]:
# Channel summary from sensor.json
for s in nusc.sensor:
    print(f"  {s['channel']:18}  modality={s['modality']}")

## 2. Synchronization verification

Each keyframe (sample) anchors on a LIDAR_TOP timestamp. For each of the 7 cameras, we matched the nearest camera frame (both sides) and required `|cam_ts - lidar_ts| ≤ 25 ms`. Keyframes failing for any channel were dropped at stage 2.

Two checks below:
1. **Raw timestamps from intermediate** — histograms of lidar↔cam diff per channel (full bag, before sync filter)
2. **Surviving samples in the dataset** — verify all sample_data within tolerance

In [ ]:
# Pick the intermediate matching the first log
log_name = nusc.log[0]['logfile']
interm = INTERMEDIATE_ROOT / log_name
print(f"intermediate: {interm}")

cam_root = interm / "cameras"
lidar_root = interm / "lidar"

cam_channels = sorted([d.name for d in cam_root.iterdir() if d.is_dir()])
ts_by_ch = {}
for ch in cam_channels:
    ts = np.array(sorted(int(f.stem) for f in (cam_root / ch).glob("*.jpg")), dtype=np.int64)
    ts_by_ch[ch] = ts
lidar_ts = np.array(
    sorted(int(f.name.split('.', 1)[0]) for f in lidar_root.glob("*.bin.zst")),
    dtype=np.int64,
)

print(f"\nlidar frames: {len(lidar_ts)}  median dt = {np.median(np.diff(lidar_ts))/1e6:.2f} ms")
for ch in cam_channels:
    ts = ts_by_ch[ch]
    print(f"  {ch:18} n={len(ts):6}  median dt = {np.median(np.diff(ts))/1e6:.2f} ms")

In [ ]:
# Per-channel nearest-neighbor diff (anchored on lidar)
diffs_by_ch = {}
for ch in cam_channels:
    cam_ts = ts_by_ch[ch]
    idx = np.searchsorted(cam_ts, lidar_ts)
    idx_l = np.clip(idx - 1, 0, len(cam_ts) - 1)
    idx_r = np.clip(idx, 0, len(cam_ts) - 1)
    d_l = np.abs(cam_ts[idx_l] - lidar_ts)
    d_r = np.abs(cam_ts[idx_r] - lidar_ts)
    diffs_by_ch[ch] = np.minimum(d_l, d_r) / 1e6  # ms

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, ch in zip(axes.flat, cam_channels):
    d = diffs_by_ch[ch]
    ax.hist(d, bins=50, range=(0, 60), color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(25, color='red', linestyle='--', label='25 ms tolerance')
    p50, p99 = np.median(d), np.percentile(d, 99)
    pct25 = 100 * np.mean(d <= 25)
    ax.set_title(f"{ch}\np50={p50:.1f}ms  p99={p99:.1f}ms  ≤25ms={pct25:.1f}%")
    ax.set_xlabel("|cam_ts − lidar_ts| (ms)")
    ax.legend()
for ax in axes.flat[len(cam_channels):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 'All cams must pass' acceptance rate per tolerance — table from sync_stats.py
anchors = lidar_ts
worst = np.zeros_like(anchors)
for ch in cam_channels:
    worst = np.maximum(worst, (diffs_by_ch[ch] * 1e6).astype(np.int64))

print(f"{'tolerance':>10}  {'kept':>10}  {'%':>7}  {'dropped':>8}")
for tol_ms in [10, 15, 20, 25, 30, 40, 50, 100]:
    kept = int(np.sum(worst <= tol_ms * 1_000_000))
    pct = 100 * kept / len(anchors)
    print(f"{tol_ms:>7} ms  {kept:>10}  {pct:6.2f}%  {len(anchors)-kept:>8}")

In [ ]:
# Surviving samples — verify all sample_data within tolerance from sample.timestamp
max_diff_per_channel = {}
for sample in nusc.sample:
    ts = sample['timestamp']  # us
    for ch, sd_token in sample['data'].items():
        sd = nusc.get('sample_data', sd_token)
        if not sd['is_key_frame']:
            continue
        d = abs(sd['timestamp'] - ts) / 1000  # ms
        if d > max_diff_per_channel.get(ch, 0):
            max_diff_per_channel[ch] = d

print("Max keyframe diff per channel across all samples (must be ≤ 25 ms):")
for ch, d in sorted(max_diff_per_channel.items()):
    flag = "OK" if d <= 25 else "FAIL"
    print(f"  {ch:18}  {d:6.2f} ms   [{flag}]")

## 3. Pose interpolation sanity

Ego pose at every sample_data timestamp was interpolated from `/novatel/oem7/odom` (50 Hz) using SLERP for rotation and linear for translation. We plot ego trajectory of the first scene to verify smoothness.

In [ ]:
scene = nusc.scene[0]
first_sample = nusc.get('sample', scene['first_sample_token'])

xs, ys, ts = [], [], []
tok = scene['first_sample_token']
while tok:
    s = nusc.get('sample', tok)
    sd = nusc.get('sample_data', s['data']['LIDAR_TOP'])
    ego = nusc.get('ego_pose', sd['ego_pose_token'])
    xs.append(ego['translation'][0])
    ys.append(ego['translation'][1])
    ts.append(ego['timestamp'])
    tok = s['next']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(xs, ys, '.-', markersize=4)
axes[0].set_aspect('equal')
axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("y (m)")
axes[0].set_title(f"{scene['name']} — ego trajectory ({len(xs)} samples)")
axes[0].grid(True, alpha=0.3)

ts = np.array(ts)
axes[1].plot(np.diff(ts) / 1000, '.-', markersize=4)
axes[1].set_xlabel("sample idx")
axes[1].set_ylabel("Δt between samples (ms)")
axes[1].set_title("Sample cadence (should be ~500 ms at 2 Hz)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Pick a sample for rendering

`nusc.render_sample` requires a real HD-map raster (devkit crops it around ego coords); our placeholder map fails the crop. We use `render_pointcloud_in_image` and `render_sample_data` instead — both work without map.

In [ ]:
# Pick mid-bag sample
sample_token = nusc.sample[len(nusc.sample) // 2]['token']
sample = nusc.get('sample', sample_token)
print(f"sample_token: {sample_token}")
print(f"timestamp:    {sample['timestamp']} us")
print(f"channels:     {sorted(sample['data'].keys())}")

## 5. LiDAR projected on a single camera (`render_pointcloud_in_image`)

In [ ]:
for cam in ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_BACK']:
    nusc.render_pointcloud_in_image(
        sample_token,
        pointsensor_channel='LIDAR_TOP',
        camera_channel=cam,
        dot_size=2,
    )

## 6. CAM_TRAFFIC — separate channel

CAM_TRAFFIC is registered as a 7th sensor channel. Its calibration is currently a placeholder (identity), so projection won't be geometrically correct yet — but the JPEG itself is in place at the matched keyframe time.

In [ ]:
import cv2
sample = nusc.get('sample', sample_token)
if 'CAM_TRAFFIC' in sample['data']:
    sd = nusc.get('sample_data', sample['data']['CAM_TRAFFIC'])
    img = cv2.imread(str(Path(DATAROOT) / sd['filename']))
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"CAM_TRAFFIC @ {sd['timestamp']} us  ({sd['filename']})")
    plt.axis('off')
    plt.show()
else:
    print("CAM_TRAFFIC not present in this dataset.")

## 7. (Optional) Render a scene inline

Compose a 6-camera 2×3 grid per sample and embed as an inline HTML5 player. No file saved. Uses `matplotlib.animation` → `to_jshtml()`, so it works headless and embeds in the notebook itself.

In [ ]:
import matplotlib.animation as manim
from IPython.display import HTML

scene = nusc.scene[0]
cell_w, cell_h = 320, 180  # tile size (smaller -> smaller notebook)
grid_cams = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']

frames = []
tok = scene['first_sample_token']
while tok:
    s = nusc.get('sample', tok)
    tiles = []
    for ch in grid_cams:
        sd = nusc.get('sample_data', s['data'][ch])
        img = cv2.imread(str(Path(DATAROOT) / sd['filename']))
        img = cv2.resize(img, (cell_w, cell_h))
        cv2.putText(img, ch, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(img, ch, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        tiles.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    top = np.hstack(tiles[:3])
    bot = np.hstack(tiles[3:])
    frames.append(np.vstack([top, bot]))
    tok = s['next']
print(f"frames collected: {len(frames)}  ({scene['name']}, ~{len(frames)/2:.1f}s @ 2 Hz)")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis('off')
im = ax.imshow(frames[0])
ax.set_title(scene['name'])

def _update(i):
    im.set_array(frames[i])
    return (im,)

ani = manim.FuncAnimation(fig, _update, frames=len(frames), interval=500, blit=True)
plt.close(fig)  # suppress static figure
HTML(ani.to_jshtml())